# Mục 3a — Khảo sát dữ liệu thô ban đầu
**Võ Minh Huy — 2311110117**

Thực hiện cùng bốn nội dung với file SQL `02_khao_sat_du_lieu_tho.sql`. Chỉ khảo sát, chưa làm sạch. Ô rỗng trong CSV tương ứng với NULL khi nhập SQL.

In [1]:
from pathlib import Path
import pandas as pd

thu_muc = Path.cwd()
if thu_muc.name == "python":
    thu_muc = thu_muc.parent
file_csv = thu_muc / "VoMinhHuy_Heart Failure Prediction_before.csv"
df = pd.read_csv(file_csv, dtype=str, keep_default_na=False)
thu_muc_ket_qua = thu_muc / "ket_qua" / "02_khao_sat_3a"
thu_muc_ket_qua.mkdir(parents=True, exist_ok=True)

tong_quan = pd.DataFrame({
    "TongSoDong": [len(df)],
    "TongSoCot": [df.shape[1]]
})
tong_quan

,TongSoDong,TongSoCot
0,309,13


## 1. Kiểu lưu trữ hiện tại
SQL dùng `nvarchar(50)`, Python đọc dưới dạng chuỗi. Cách lưu khác nhau nhưng cùng mục tiêu giữ nguyên giá trị lỗi. Đây chưa phải kiểu logic sau chuẩn hóa. Python dùng chuỗi rỗng thay cho NULL để giữ nguyên ô rỗng từ CSV.

In [2]:
kieu_du_lieu = pd.DataFrame({
    "STT": range(1, df.shape[1] + 1),
    "TenCot": df.columns,
    "KieuLuuTruPython": [str(df[c].dtype) for c in df.columns],
    "CachDoc": "Chuỗi, giữ ô rỗng"
})
kieu_du_lieu

,STT,TenCot,KieuLuuTruPython,CachDoc
0,1,age,str,"Chuỗi, giữ ô rỗng"
1,2,anaemia,str,"Chuỗi, giữ ô rỗng"
2,3,creatinine_phosphokinase,str,"Chuỗi, giữ ô rỗng"
3,4,diabetes,str,"Chuỗi, giữ ô rỗng"
4,5,ejection_fraction,str,"Chuỗi, giữ ô rỗng"
5,6,high_blood_pressure,str,"Chuỗi, giữ ô rỗng"
6,7,platelets,str,"Chuỗi, giữ ô rỗng"
7,8,serum_creatinine,str,"Chuỗi, giữ ô rỗng"
8,9,serum_sodium,str,"Chuỗi, giữ ô rỗng"
9,10,sex,str,"Chuỗi, giữ ô rỗng"


## 2. Giá trị thiếu ở từng cột
Đếm trên toàn bộ 309 dòng, bao gồm 5 dòng trống. Không xóa dòng hoặc sửa giá trị. Mẫu số tính tỷ lệ là 309 cho mỗi cột.

In [3]:
mat_na_thieu = df.apply(lambda cot: cot.str.strip().eq(""))
so_thieu = mat_na_thieu.sum()
thong_ke_thieu = pd.DataFrame({
    "STT": range(1, df.shape[1] + 1),
    "TenCot": df.columns,
    "SoGiaTriThieu": so_thieu.to_numpy(),
    "TyLeThieuPhanTram": (100 * so_thieu / len(df)).round(2).to_numpy()
})
thong_ke_thieu

,STT,TenCot,SoGiaTriThieu,TyLeThieuPhanTram
0,1,age,26,8.41
1,2,anaemia,5,1.62
2,3,creatinine_phosphokinase,25,8.09
3,4,diabetes,5,1.62
4,5,ejection_fraction,5,1.62
5,6,high_blood_pressure,5,1.62
6,7,platelets,25,8.09
7,8,serum_creatinine,25,8.09
8,9,serum_sodium,26,8.41
9,10,sex,25,8.09


## 3. Tần suất từng cách ghi của các biến nhị phân
Giữ riêng các giá trị có khoảng trắng hoặc cách viết khác nhau để phát hiện không nhất quán. `<NULL>` biểu diễn ô rỗng; `SoKyTu` tính cả khoảng trắng đầu và cuối. DEATH_EVENT là nhãn kết quả, được khảo sát nhưng không gộp vào đặc trưng đầu vào.

In [4]:
cot_nhi_phan = [
    "anaemia", "diabetes", "high_blood_pressure",
    "sex", "smoking", "DEATH_EVENT"
]
dong_tan_suat = []
for stt, cot in enumerate(cot_nhi_phan, start=1):
    for gia_tri, so_luong in df[cot].value_counts(dropna=False).items():
        dong_tan_suat.append({
            "STT": stt,
            "TenCot": cot,
            "GiaTriHienThi": "<NULL>" if gia_tri == "" else "[" + gia_tri + "]",
            "SoKyTu": None if gia_tri == "" else len(gia_tri),
            "SoLuong": int(so_luong)
        })
tan_suat = pd.DataFrame(dong_tan_suat).sort_values(
    ["STT", "GiaTriHienThi"], ignore_index=True
)
tan_suat["SoKyTu"] = tan_suat["SoKyTu"].astype("Int64")
tan_suat

,STT,TenCot,GiaTriHienThi,SoKyTu,SoLuong
0,1,anaemia,<NULL>,<NA>,5
1,1,anaemia,[0],1,170
2,1,anaemia,[1],1,134
3,2,diabetes,<NULL>,<NA>,5
4,2,diabetes,[0],1,179
5,2,diabetes,[1],1,125
6,3,high_blood_pressure,<NULL>,<NA>,5
7,3,high_blood_pressure,[0],1,197
8,3,high_blood_pressure,[1],1,107
9,4,sex,<NULL>,<NA>,25


## 4. Nhận xét tập trung vào các giá trị không nhất quán
`sex` có mã 0, 1 xen lẫn m, Male, F, female và khoảng trắng; chưa chuẩn hóa. `DEATH_EVENT` có giá trị 2 ngoài hai nhãn 0 và 1. Không tự sửa các giá trị này trong mục 3a. Hai bảng dưới đây thuận tiện để chụp minh chứng.

In [5]:
tan_suat.loc[tan_suat["TenCot"] == "sex"].reset_index(drop=True)

,STT,TenCot,GiaTriHienThi,SoKyTu,SoLuong
0,4,sex,<NULL>,<NA>,25
1,4,sex,[ F ],7,1
2,4,sex,[ Male ],10,24
3,4,sex,[0],1,82
4,4,sex,[1],1,160
5,4,sex,[female],6,16
6,4,sex,[m],1,1


In [6]:
tan_suat.loc[tan_suat["TenCot"] == "DEATH_EVENT"].reset_index(drop=True)

,STT,TenCot,GiaTriHienThi,SoKyTu,SoLuong
0,6,DEATH_EVENT,<NULL>,<NA>,5
1,6,DEATH_EVENT,[0],1,204
2,6,DEATH_EVENT,[1],1,99
3,6,DEATH_EVENT,[2],1,1


## 5. Lưu kết quả khảo sát
Chỉ xuất bảng thống kê; không ghi đè CSV nguồn hoặc tạo after ở bước này.

In [7]:
tong_quan.to_csv(thu_muc_ket_qua / "tong_quan_python.csv", index=False, encoding="utf-8-sig")
kieu_du_lieu.to_csv(thu_muc_ket_qua / "kieu_du_lieu_python.csv", index=False, encoding="utf-8-sig")
thong_ke_thieu.to_csv(thu_muc_ket_qua / "gia_tri_thieu_python.csv", index=False, encoding="utf-8-sig")
tan_suat.to_csv(thu_muc_ket_qua / "tan_suat_python.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"TrangThai": ["Đã lưu 4 bảng thống kê; giữ nguyên dữ liệu nguồn."]})

,TrangThai
0,Đã lưu 4 bảng thống kê; giữ nguyên dữ liệu nguồn.
